# OASIS-2 Multimodal Training Pipeline

This notebook is the entry point for the **Multimodal Alzheimer's AI Monitoring Platform**.
It fuses 3D brain MRI scans with clinical metadata (Age, Sex, MMSE) using the `OASISMultimodalDenseNet` architecture.

In [5]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Constants for your environment
DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_multimodal_v1' # Fresh multimodal experiment

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(f"{name}: {'[OK]' if path.exists() else '[MISSING]'} {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [6]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = '95132f8' # Latest optimized commit

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

# Clean up any stale repo clones
for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print(f'Active commit: {repo_commit}')

RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense
Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt

RUNNING git-rev-parse: git rev-parse HEAD
6cce44fa4c2bb3cf392ed9b1a12abeb0d8bb2225

Active commit: 6cce44fa4c2bb3cf392ed9b1a12abeb0d8bb2225


### Start Multimodal Training
Initiates a 50-epoch training run fusing MRI volumes with clinical metadata (Age, Sex, MMSE).

In [9]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Change runtime type to T4 GPU before proceeding.")

os.chdir(BACKEND_ROOT)

!python scripts/train_oasis2_colab.py \
    --project-root {BACKEND_ROOT} \
    --runtime-root {RUNTIME_ROOT} \
    --bundle-root {OASIS2_BUNDLE_ROOT} \
    --run-name {RUN_NAME} \
    --epochs 50 \
    --batch-size 2 \
    --gradient-accumulation-steps 4 \
    --num-workers 2 \
    --image-size 96 96 96 \
    --learning-rate 2e-4 \
    --weight-decay 0.01 \
    --scheduler cosine \
    --weighted-sampling \
    --seed 42 \
    --split-seed 42 \
    --device auto \
    --config configs/oasis2_train.yaml

Traceback (most recent call last):
  File "/content/cerebrasense/alz_backend/scripts/train_oasis2_colab.py", line 27, in <module>
    from scripts.train_oasis2 import apply_cli_overrides, build_parser as build_train_parser  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerebrasense/alz_backend/scripts/train_oasis2.py", line 14, in <module>
    from src.training.oasis2_research import default_oasis2_train_config_path, run_research_oasis2_training  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerebrasense/alz_backend/src/training/__init__.py", line 3, in <module>
    from .train_kaggle import KaggleTrainingConfig, build_kaggle_monai_training_components, build_kaggle_training_run_name
  File "/content/cerebrasense/alz_backend/src/training/train_kaggle.py", line 10, in <module>
    from src.models.kaggle_model import Kaggl

### Result Summary
View final metrics after training completes.

In [8]:
import json
import pandas as pd
from pathlib import Path

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"\nTraining Summary for {RUN_NAME}:")
    print(f"- Best Epoch: {summary.get('best_epoch')}")
    print(f"- Best Val AUROC: {summary.get('best_monitor_value'):.4f}")
    print(f"- Best Checkpoint: {summary.get('best_checkpoint')}")

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print("\nLast 5 Epochs:")
    print(df.tail())



Last 5 Epochs:
   epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
4      5       0.000191    0.217896  0.212220  0.555556  0.574074   0.548387   
5      6       0.000187    0.216309  0.213154  0.481481  0.529492   0.487179   
6      7       0.000181    0.217416  0.213828  0.518519  0.560357   0.511628   
7      8       0.000174    0.216499  0.223254  0.518519  0.574074   0.509434   
8      9       0.000167    0.213120  0.242731  0.518519  0.565158   0.509434   

     recall        f1  sensitivity  specificity  train_batches  val_batches  
4  0.629630  0.586207     0.629630     0.481481            130           27  
5  0.703704  0.575758     0.703704     0.259259            130           27  
6  0.814815  0.628571     0.814815     0.222222            130           27  
7  1.000000  0.675000     1.000000     0.037037            130           27  
8  1.000000  0.675000     1.000000     0.037037            130           27  
